# script to determine with absolute head and tail position(measured with dlc) 
## based on the head and tail positions I determine the the concentration of head and tail based on the gradient created with the curve_fit script

In [ ]:
#Importing the toolbox (takes several seconds)
import pandas as pd
from pathlib import Path
import numpy as np
import os
import matplotlib.pyplot as plt
import tifffile as tiff
from natsort import natsorted
import glob
import matplotlib.patches as patches
from coordinate_conversion_functions import *
from track_plotting_daniel import *
import fnmatch

## read in data

#### these are the center coordinates

In [ ]:
path_center_coords='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/all_tracks/2020-07-01_17-01-27_chemotaxis_worm6-TablePosRecord.txt'
df_center_coords=pd.read_csv(path_center_coords)

#### this are the head and tail positions measured via dlc

In [ ]:
path_head_tail='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/avi_all/2020-07-01_17-01-27_chemotaxis_worm6-channel-0-bigtiffDLC_resnet50_HeadTailAug10shuffle1_275000_filtered.h5'
df_head_tail_coords=pd.read_hdf(path_head_tail, low_memory=False)
#df_head_tail_coords

# get absolute head and tail coordinates

#### formate the dataframe and rename columns

In [ ]:
scorer=df_head_tail_coords.columns.get_level_values(0)[0] # defines scorer as the first column of the dataframe",
df_head_tail_center_coords=df_center_coords
df_head_tail_center_coords=df_head_tail_center_coords.rename(columns = {'x': 'x_center',
                'y': 'y_center'}, inplace = False)
#df_head_tail_center_coords.head()

#### defining the parameters for the function to calulate the corrected head and tail positions

In [ ]:
px_mm=0.00325 # this value represents how much mm is one pixel
x_lenght_frame=608 #lenght of x axis in frame (measured with fiji)
y_width_frame=610 #width of the y axis in frame
center_x=df_head_tail_center_coords['x_center'] #absolute coordinates of center position x
center_y=df_head_tail_center_coords['y_center'] #absolute coordinates of center position y
data_head_x=df_head_tail_coords[scorer]['Head']['x'].values #getting the x position of head
data_head_y=df_head_tail_coords[scorer]['Head']['y'].values #getting the y position of head
data_tail_x=df_head_tail_coords[scorer]['Tail']['x'].values #getting the x position of tail
data_tail_y=df_head_tail_coords[scorer]['Tail']['y'].values #getting the y position of tail

#### calculate absolute head and tail positon

In [ ]:
#create absolute coordinates for head
corrected_head_position=corrected_absolute_coordinates(center_x,center_y,data_head_x,
                data_head_y,px_mm,y_width_frame,x_lenght_frame)

corrected_head_position= corrected_head_position.rename(columns = {'x_corrected': 'x_head_corrected',
                'y_corrected': 'y_head_corrected'}, inplace = False)

corrected_tail_position=corrected_absolute_coordinates(center_x,center_y,data_tail_x,
                                                       data_tail_y,px_mm,y_width_frame,x_lenght_frame)
#create absolute coordinates for tail
corrected_tail_position= corrected_tail_position.rename(columns = {'x_corrected': 'x_tail_corrected',
                'y_corrected': 'y_tail_corrected'}, inplace = False)

#### joining dfs for center coords and corrected head and tail coordinates

In [ ]:
#joining dfs for center coords and corrected head and tail coordinates
df_head_tail_center_coords = pd.concat([df_head_tail_center_coords,corrected_head_position, 
                                        corrected_tail_position], axis=1)
#df_head_tail_center_coords

# correct for food position
(measured manually for each recording)

In [ ]:
#corrected for food position
xref=9.10
df_head_tail_center_coords['x_tail_corrected']=df_head_tail_center_coords['x_tail_corrected']-xref
df_head_tail_center_coords['x_head_corrected']=df_head_tail_center_coords['x_head_corrected']-xref
df_head_tail_center_coords['x_center']=df_head_tail_center_coords['x_center']-xref
df_head_tail_center_coords.head()

# calculate concentration change

In [ ]:
#this is telling you the concentration of every x position occuring in the data
#since it is a stripe assay only the x coordinates matter for concentration change
#parameters determined via the curve fit script
L=1
x0=3.81953189
k=-0.69421567
b=0
x_head_corrected=df_head_tail_center_coords['x_head_corrected'].values 
x_tail_corrected=df_head_tail_center_coords['x_tail_corrected'].values
x_center=df_head_tail_center_coords['x_center'].values
y_head_corrected=df_head_tail_center_coords['y_head_corrected'].values 
y_tail_corrected=df_head_tail_center_coords['y_tail_corrected'].values
y_center=df_head_tail_center_coords['y_center'].values
y_fit_head=sigmoid(x_head_corrected, L, x0, k, b)
y_fit_tail=sigmoid(x_tail_corrected, L, x0, k, b)
y_fit_center=sigmoid(x_center, L, x0, k, b)

#### add concentration data to the coordinate dataframe

In [ ]:
#creating df with x y head/tail position and the corresponding concentration
position_concentration = pd.DataFrame(data=x_head_corrected,columns=["x_head_corrected"])
position_concentration['y_head_corrected']=pd.DataFrame(data=y_head_corrected,columns=["y_head_corrected"])
position_concentration['x_tail_corrected']=pd.DataFrame(data=x_tail_corrected,columns=["x_tail_corrected"])
position_concentration['y_tail_corrected']=pd.DataFrame(data=y_tail_corrected,columns=["y_tail_corrected"])
position_concentration['x_center']=pd.DataFrame(data=x_center,columns=["x_center"])
position_concentration['y_center']=pd.DataFrame(data=y_center,columns=["y_center"])
position_concentration['concentration_head']=y_fit_head
position_concentration['concentration_tail']=y_fit_tail
position_concentration['concentration_center']=y_fit_center
position_concentration.head()

#### calculate concentration change from one position to the next

In [ ]:
#calculating concentration change
fps=167 #frames per second
concentration_change=position_concentration
concentration_change["concentration_change_head"] = position_concentration["concentration_head"].diff()
concentration_change["concentration_change_tail"] = position_concentration["concentration_tail"].diff()
concentration_change["concentration_change_center"] = position_concentration["concentration_center"].diff()
concentration_change['seconds']=np.arange(0, len(concentration_change)/fps,1/fps)
concentration_change.head()

# plot track

In [ ]:
#plot track of head tail and center
line_width=0.5
head_conc=concentration_change['concentration_head']
tail_conc=concentration_change['concentration_tail']
center_conc=concentration_change['concentration_center']

head_pos_x=concentration_change['x_head_corrected']
tail_pos_x=concentration_change['x_tail_corrected']
center_pos_x=concentration_change['x_center']

head_pos_y=df_head_tail_center_coords['y_head_corrected']
tail_pos_y=df_head_tail_center_coords['y_tail_corrected']
center_pos_y=df_head_tail_center_coords['y_center']

fig3, ax3 = plt.subplots(1,1, figsize = (10,5), dpi=600)
#ax3.scatter(head_pos_x,head_pos_y,c=head_conc,s=0.01)
#ax3.scatter(tail_pos_x,tail_pos_y,c=tail_conc,s=0.01)
#ax3.scatter(center_pos_x,center_pos_y,c=center_conc,s=0.01)
ax3.plot(head_pos_x,head_pos_y,label="head",linewidth=line_width,)
ax3.plot(tail_pos_x,tail_pos_y,linewidth=line_width,label="tail")
ax3.plot(center_pos_x,center_pos_y,linewidth=line_width,label="center")
ax3.axvline(x=0, ymin=0, ymax=1, lw=10, alpha=.5, color='y')
ax3.legend()
plt.xlim(-1,44)
plt.xlabel('X position')
plt.ylabel('Y position')
plt.show()

# plot cumulative concentration

In [ ]:
#concentraton as a function of time
line_width=0.5
x=concentration_change['seconds']
head_conc=concentration_change['concentration_head']
tail_conc=concentration_change['concentration_tail']
center_conc=concentration_change['concentration_center']
xfig3, ax3 = plt.subplots(1,1, figsize = (10,5), dpi=800)
ax3.plot(x,head_conc,label="head",linewidth=line_width)
ax3.plot(x,tail_conc,label="tail",linewidth=line_width)
ax3.plot(x,center_conc,label="center",linewidth=line_width)
ax3.legend()
plt.show()

# create csv files with absolute head tail coordinates for all worms

In [ ]:
#defining the parameters for the function to calulate the corrected head and tail positions
px_mm=0.00325 # this value represents how much mm is one pixel
x_lenght_frame=608 #lenght of x axis in frame (measured with fiji)
y_width_frame=610 #width of the y axis in frame

In [ ]:
#loop to create csv with absolute heasd and tail coordinates + correction for xref
path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/'
all_x_ref=pd.read_csv('/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/all_x_refs.csv')
for i, folder in enumerate(natsorted(os.listdir(path))):
    if i>len(all_x_ref)-1:break
    x_ref=all_x_ref['x_ref'][i] 
    print(i, ': x ref of ', folder, 'is: ', x_ref)
    hdf5_files=glob.glob(os.path.join(path,folder,'*.h5'))
    txt_files=glob.glob(os.path.join(path,folder,'*.txt'))
    #acces the first (and should be only hdf5 file)
    hdf5_file=hdf5_files[0]
    df_head_tail=pd.read_hdf(hdf5_file)
    #print(df_head_tail)
    scorer= df_head_tail.columns.get_level_values(0)[0]
    data_head_x= df_head_tail[scorer]['Head']['x'].values #getting the x position of head
    data_head_y=df_head_tail[scorer]['Head']['y'].values #getting the y position of head
    data_tail_x=df_head_tail[scorer]['Tail']['x'].values #getting the x position of tail
    data_tail_y=df_head_tail[scorer]['Tail']['y'].values #getting the y position of tail
    txt_file=txt_files[0]
    df_center_coords=pd.read_csv(txt_file)
    #print(df_center_coords)
    if len(df_center_coords) > len(df_head_tail):
        rows_diff=len(df_center_coords) - len(df_head_tail)
        print(txt_files,'is',rows_diff,' rows longer')
        df_center_coords.drop(df_center_coords.tail(rows_diff).index,inplace=True)
    df_center_coords = df_center_coords[0:200001] #some center coordinates have more rows than dlc data
    #print(df_center_coords)
    df_center_coords = df_center_coords.rename(columns={"x":"x_center"})
    df_center_coords = df_center_coords.rename(columns={"y":"y_center"})
    #print(df_center_coords)
    center_x=df_center_coords['x_center'].values #absolute coordinates of center position x
    center_y=df_center_coords['y_center'].values #absolute coordinates of center position y
    #print(df_head_tail)
    corrected_head=corrected_absolute_coordinates(center_x,center_y,data_head_x, data_head_y,px_mm,y_width_frame,x_lenght_frame)  #fucntion to calculate absolute x coordinates for head and tial
    corrected_tail=corrected_absolute_coordinates(center_x,center_y,data_tail_x,data_tail_y,px_mm,y_width_frame,x_lenght_frame)
    corrected_head = corrected_head.rename(columns={"x_corrected":"x_head_corrected"}) #rename to have differnt column names for head and tail when the df is put togeether
    corrected_head = corrected_head.rename(columns={"y_corrected":"y_head_corrected"})
    corrected_tail = corrected_tail.rename(columns={"x_corrected":"x_tail_corrected"})
    corrected_tail = corrected_tail.rename(columns={"y_corrected":"y_tail_corrected"})
    df_head_tail_center_coords=pd.concat([df_center_coords,corrected_head,corrected_tail], axis=1)
    df_head_tail_center_coords['x_head_corrected']=df_head_tail_center_coords['x_head_corrected']-x_ref  #correcting for xref
    df_head_tail_center_coords['x_tail_corrected']=df_head_tail_center_coords['x_tail_corrected']-x_ref
    df_head_tail_center_coords['x_center']=df_head_tail_center_coords['x_center']-x_ref
    #print(df_head_tail_center_coords)
    #df_head_tail_center_coords.to_csv('/groups/zimmer/shared_projects/DanielMitic/data/ulises_chemotaxis/'+folder+'.csv')

# create csv files with head tail coordinates + concentration for all worms

In [ ]:
#this is telling you the concentration of every x position occuring in the data
#parameters determined via the curve fit script
L=1
x0=3.81953189
k=-0.69421567
b=0

In [ ]:
#creates csv files with concentration data
path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/head_tail_center_absolute_coordinates_x_ref_corrected/'
for filename in natsorted(os.listdir(path)):
    if filename.endswith('.csv'):
        df_head_tail_center_coords=pd.read_csv(path+filename)
        x_head_corrected=df_head_tail_center_coords['x_head_corrected'].values #xref adust gradient for food position
        x_tail_corrected=df_head_tail_center_coords['x_tail_corrected'].values
        y_head_corrected=df_head_tail_center_coords['y_head_corrected'].values 
        y_tail_corrected=df_head_tail_center_coords['y_tail_corrected'].values
        y_center=df_head_tail_center_coords['y_center'].values
        x_center=df_head_tail_center_coords['x_center'].values
        y_fit_head=sigmoid(x_head_corrected, L, x0, k, b)
        y_fit_tail=sigmoid(x_tail_corrected, L, x0, k, b)
        y_fit_center=sigmoid(x_center, L, x0, k, b)
        #creating df with x/y head/tail position and the corresponding concentration
        position_concentration = pd.DataFrame(data=x_head_corrected,columns=["x_head_corrected"])
        position_concentration['y_head_corrected']=pd.DataFrame(data=y_head_corrected,columns=["y_head_corrected"])
        position_concentration['x_tail_corrected']=pd.DataFrame(data=x_tail_corrected,columns=["x_tail_corrected"])
        position_concentration['y_tail_corrected']=pd.DataFrame(data=y_tail_corrected,columns=["y_tail_corrected"])
        position_concentration['x_center']=pd.DataFrame(data=x_center,columns=["x_center"])
        position_concentration['y_center']=pd.DataFrame(data=y_center,columns=["y_center"])
        position_concentration['concentration_head']=y_fit_head
        position_concentration['concentration_tail']=y_fit_tail
        position_concentration['concentration_center']=y_fit_center
        #calculating concentration change
        concentration_change=position_concentration
        concentration_change["concentration_change_head"] = position_concentration["concentration_head"].diff()
        concentration_change["concentration_change_tail"] = position_concentration["concentration_tail"].diff()
        concentration_change["concentration_change_center"] = position_concentration["concentration_center"].diff()
        concentration_change['seconds']=np.arange(0, len(concentration_change)/fps,1/fps)
        #print(concentration_change)
        #concentration_change.to_csv('/groups/zimmer//Daniel_Mitic/data/ulises_chemotaxis/'+filename+'concentration.csv', index=False)

# plot all tracks of all worms

In [ ]:
figsize_x=10
figsize_y=5
line_width=0.1
#gives a key error for y_head corrected but code works
path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/head_tail_center_absolute_coordinates_x_ref_corrected_concentration/'
#plot all tracks for center head and tail
for filename in natsorted(os.listdir(path)):
    plot_tracks_center_head_tail(path+filename,figsize_x,figsize_y,line_width)
    plt.title(filename)
    plt.xlabel('X position')
    plt.ylabel('Y posiiton')
    #plt.savefig(path+filename+'.png')

## plot concentration over time

In [ ]:
#gives following error but code works UnicodeDecodeError: 'utf-8' codec can't decode byte 0x89 in position 0: invalid start byte
#concentraton and time for all tracks
path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/head_tail_center_absolute_coordinates_x_ref_corrected_concentration/'
line_width=0.1
x=np.arange(200001)

for filename in natsorted(os.listdir(path)):
    if fnmatch.fnmatch(filename, '*.csv'):
        concentration_change=pd.read_csv(path+filename)
        head_conc=concentration_change['concentration_head']
        tail_conc=concentration_change['concentration_tail']
        center_conc=concentration_change['concentration_center']

        xfig3, ax3 = plt.subplots(1,1, figsize = (10,5), dpi=600)
        ax3.plot(x,head_conc,label="head",linewidth=line_width)
        ax3.plot(x,tail_conc,label="tail",linewidth=line_width)
        ax3.plot(x,center_conc,label="center",linewidth=line_width)
        plt.title(filename)
        plt.xlabel('time')
        plt.legend()
        plt.ylabel('concentration')
        plt.ylim([0,1])
        #plt.savefig(path+filename+'.png')

# histogram with concentration changes for all worms

In [ ]:
#histogram concentration change for all worms
path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/head_tail_center_absolute_coordinates_x_ref_corrected_concentration/'
output_path='/groups/zimmer/shared_projects/DanielMitic/data/ulises_chemotaxis/'
bins = np.linspace(-0.0015, 0.0015, 500)
for filename in natsorted(os.listdir(path)):
    if fnmatch.fnmatch(filename, '*.csv'):
        print(filename)
        concentration_change=pd.read_csv(path+filename)
        head_conc=concentration_change['concentration_change_head']
        tail_conc=concentration_change['concentration_change_tail']
        center_conc=concentration_change['concentration_change_center']
        plt.xticks(np.arange(-0.0015, 0.0015, step=0.00005)) 
        plt.hist(head_conc, bins, alpha=0.5, label='head_conc')
        plt.hist(tail_conc, bins, alpha=0.5, label='tail_conc')
        plt.hist(center_conc, bins, alpha=0.5, label='center_conc')
        plt.legend(loc='upper right')
        plt.ylim(0, 10000)
        plt.xlim(-0.00015, 0.00015)
        #plt.savefig(output_path+filename+'.png')
        plt.show()

# plot specific parts of tracks

In [ ]:
path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/head_tail_center_absolute_coordinates_x_ref_corrected_concentration/2020-07-01_14-41-11_chemotaxisl_worm2-TablePosRecord.csvconcentration.csv'
concentration_change=pd.read_csv(path, low_memory=False)
#concentration_change

In [ ]:
begin_x=2
end_x=4
begin_y=22
end_y=24.88
position=position_to_time(concentration_change,begin_x,end_x,begin_y,end_y)
plot_track_section(concentration_change,position)

In [ ]:
plot_concentration_section(concentration_change,position)

## here i tried to remove outliers

In [ ]:
conc_change_head=concentration_change['concentration_change_head']

concentration_change_outliers_out_head=concentration_change[np.abs(conc_change_head-conc_change_head.mean())
                                                       <= (3*conc_change_head.std())] 
#for tail
conc_change_tail=concentration_change_outliers_out_head['concentration_change_tail']

concentration_change_outliers_out=concentration_change_outliers_out_head[np.abs(conc_change_tail-conc_change_tail.mean())
                                                       <= (3*conc_change_tail.std())]


In [ ]:
concentration_change_outliers_out_head[np.abs(conc_change_tail-conc_change_tail.mean())
                                                       <= (3*conc_change_tail.std())]

In [ ]:
#https://stackoverflow.com/questions/23199796/detect-and-exclude-outliers-in-pandas-data-frame
conc_change_head=concentration_change['concentration_change_head']

conc_extremes=concentration_change[np.abs(conc_change_head-conc_change_head.mean())
                                                       >= (3*conc_change_head.std())] 
#for tail
conc_change_tail=concentration_change['concentration_change_tail']

conc_extremes_tail=concentration_change[np.abs(conc_change_tail-conc_change_tail.mean())
                                                     >= (3*conc_change_tail.std())]

In [ ]:
#create list to plot the position of outliers in track

#locates coordinates of extreme values within the df_head_tail_coords where all x and y positions are strored
extreme_values=concentration_change.loc[concentration_change['x_head_corrected'].isin(conc_extremes['x_head_corrected'].values)]
x_extreme=extreme_values['x_head_corrected'].tolist()
y_extreme=extreme_values['y_head_corrected'].tolist()

#tail
extreme_values_tail=concentration_change.loc[concentration_change['x_tail_corrected'].isin(conc_extremes_tail['x_tail_corrected'].values)]
x_extreme_tail=extreme_values_tail['x_tail_corrected'].tolist()
y_extreme_tail=extreme_values_tail['y_tail_corrected'].tolist()


In [ ]:
#show outliers in track
figsize_x=10
figsize_y=5
line_width=0.1

#plot track
head_pos_x=concentration_change['x_head_corrected']
tail_pos_x=concentration_change['x_tail_corrected']
center_pos_x=concentration_change['x_center']

head_pos_y=concentration_change['y_head_corrected']
tail_pos_y=concentration_change['y_tail_corrected']
center_pos_y=concentration_change['y_center']

fig3, ax3 = plt.subplots(1,1, figsize = (figsize_x,figsize_y), dpi=600)
ax3.plot(head_pos_x,head_pos_y,linewidth=line_width,label='head')
ax3.plot(tail_pos_x,tail_pos_y,linewidth=line_width,label='tail')
ax3.plot(center_pos_x,center_pos_y,linewidth=line_width,label='center')
ax3.axvline(x=0, ymin=0, ymax=1, lw=10, alpha=.5, color='y')
plt.scatter([x_extreme], [y_extreme],s=10,facecolors='none',edgecolors='b')
plt.scatter([x_extreme_tail], [y_extreme_tail],s=10,facecolors='none',edgecolors='r')
plt.legend()
#plt.savefig('chemotaxis_worm2_conc_change_outliers.png')
plt.show()

In [ ]:
#histogram with outliers
head_conc=concentration_change['concentration_change_head']
tail_conc=concentration_change['concentration_change_tail']
center_conc=concentration_change['concentration_change_center']
head_conc=head_conc.tolist()
tail_conc=tail_conc.tolist()
center_conc=center_conc.tolist()

bins = np.linspace(-0.0015, 0.0015, 500)
plt.xticks(np.arange(-0.0015, 0.0015, step=0.00005)) 

plt.hist(head_conc, bins, alpha=0.5, label='head_conc')
plt.hist(tail_conc, bins, alpha=0.5, label='tail_conc')
plt.hist(center_conc, bins, alpha=0.5, label='center_conc')
plt.legend(loc='upper right')
plt.ylim(0, 10000)
plt.xlim(-0.00015, 0.00015)
plt.savefig('chemotaxis_worm2_concentration_change_histogram')
plt.show()

In [ ]:
#removing outliers (values bigger then 3xstd)
#for head
conc_change_head=concentration_change['concentration_change_head']

concentration_change_outliers_out_head=concentration_change[np.abs(conc_change_head-conc_change_head.mean())
                                                       <= (3*conc_change_head.std())] 
#for tail
conc_change_tail=concentration_change_outliers_out_head['concentration_change_tail']

concentration_change_outliers_out=concentration_change_outliers_out_head[np.abs(conc_change_tail-conc_change_tail.mean())
                                                       <= (3*conc_change_tail.std())]
concentration_change_outliers_out


In [ ]:
#concentraiton change plot without outliers
#concentraton_change and time
x=np.arange(199935)
head_conc=concentration_change_outliers_out['concentration_change_head']
tail_conc=concentration_change_outliers_out['concentration_change_tail']
center_conc=concentration_change_outliers_out['concentration_change_center']

head_pos_x=concentration_change_outliers_out['x_head_corrected']
tail_pos_x=concentration_change_outliers_out['x_tail_corrected']
center_pos_x=concentration_change_outliers_out['x_center']
xfig3, ax3 = plt.subplots(1,1, figsize = (10,5), dpi=600)
ax3.plot(x,head_conc,label="head")
ax3.plot(x,tail_conc,label="tail")
ax3.plot(x,center_conc,label="center")


In [ ]:
#define values for histogram without outliers
head_conc=concentration_change_outliers_out['concentration_change_head']
tail_conc=concentration_change_outliers_out['concentration_change_tail']
center_conc=concentration_change_outliers_out['concentration_change_center']
head_conc=head_conc.tolist()
tail_conc=tail_conc.tolist()
center_conc=center_conc.tolist()

In [ ]:
#conc_change histogram without outliers
bins = np.linspace(-0.0015, 0.0015, 500)
plt.xticks(np.arange(-0.0015, 0.0015, step=0.00005)) 

plt.hist(head_conc, bins, alpha=0.5, label='head_conc')
plt.hist(tail_conc, bins, alpha=0.5, label='tail_conc')
plt.hist(center_conc, bins, alpha=0.5, label='center_conc')
plt.legend(loc='upper right')
plt.ylim(0, 10000)
plt.xlim(-0.00015, 0.00015)
plt.show()
